# Broadcast Variables: Efficient Lookup Distribution via P2P

**Module**: 06_memory_and_storage  
**Notebook**: 06_broadcast_variables.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How broadcast variables work, BitTorrent-like peer-to-peer distribution, and sharing read-only lookup structures across executor tasks.
- **Why does this topic exist?**: Broadcasting copies data to each executor once instead of shipping it with every individual task closure.
- **What problem does it solve?**: Serializing a 50 MB lookup dictionary inside every task saturates driver network.
- **Where is it used in real-world Data Engineering?**: Enriching streaming data with reference tables.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Executors, serialization, memory model |
| **Required depth** | Working knowledge: can use broadcast variables and explain the distribution mechanism |
| **Data Engineer** | Share read-only lookup data efficiently without shipping a copy with every task. |
| **Senior Data Engineer** | Decide when a broadcast variable is appropriate and how to size it. |
| **Spark specialist** | Reason about TorrentBroadcast, block replication, and driver memory. |
| **Next topic** | `06_memory_and_storage/07_broadcast_join.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Without a broadcast variable, a large lookup object captured by a closure is serialized with every task.
Broadcast variables send the object to each executor once, then tasks read it locally. The mechanism reduces
network and serialization overhead, but the object must fit in executor memory and must be read-only.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Broadcast variable** | Read-only shared object | Distributed once per executor and cached |
| **TorrentBroadcast** | Spark's broadcast implementation | Chunked distribution with peer-to-peer sharing |
| **Closure** | Code and captured variables sent to tasks | Large captured objects are expensive |
| **Block manager** | Stores broadcast blocks | Serves local reads to tasks |
| **Driver memory** | Holds the source object | Must accommodate the object before broadcast |
| **Immutability** | Broadcast values cannot be modified | Updates require a new broadcast |

### How it works, step by step

1. The driver serializes the object and splits it into chunks.
2. The driver registers the broadcast with the block manager.
3. Executors fetch chunks, initially from the driver and then from peers.
4. Each executor assembles and caches a local copy.
5. Tasks read the local copy instead of receiving it in their closure.
6. The broadcast remains until it is destroyed or the executor exits.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Broadcast Variable Architecture

- Driver serializes object once.
- Distributed to executors via P2P BitTorrent protocol.
- Cached in executor BlockManager for all task threads.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Broadcast_Variables_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Create lookup dictionary on Driver
currency_rates = {"USD": 1.0, "EUR": 0.92, "GBP": 0.79, "EGP": 50.0}

# Broadcast to all worker executors
b_rates = spark.sparkContext.broadcast(currency_rates)
print(f"Broadcast variable ID: {b_rates.id}")
print(f"Broadcast value lookup: USD -> {b_rates.value['USD']}")

# Use inside transformation
sample_rdd = spark.sparkContext.parallelize([("USD", 100), ("EUR", 50), ("EGP", 1000)])
converted = sample_rdd.map(lambda pair: (pair[0], pair[1] / b_rates.value.get(pair[0], 1.0))).collect()
print("Converted amounts:", converted)

---

## Experiment: Unpersisting Broadcast Variable

- **1. What are we testing?**: Call unpersist() and destroy() on broadcast variable.
- **2. What do we expect?**: Releases blocks from executor BlockManagers.

In [ ]:
b_rates.unpersist(); print('Broadcast unpersisted successfully')

**3. What happened?**: Freed executor memory.  
**4. Why did it happen?**: unpersist() removes blocks from workers.  
**5. Practical Engineering Lesson**: Unpersist large broadcast variables when no longer needed.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Country and currency lookup

- **Industry**: Payments
- **Scenario**: Every task needs a small table of country codes and currency rules.
- **Problem**: Capturing the table in a closure serializes it for every task, adding overhead and network traffic.
- **Why this concept is relevant**: A broadcast variable ships the table once per executor and lets all tasks share it.
- **Implementation idea**: Broadcast the lookup, use it inside a UDF or map, and measure serialized task size.
- **Expected result**: Lower task dispatch overhead and faster execution.
- **Alternative**: A join against a small dimension table.
- **Trade-offs**: A broadcast join is often simpler and lets the optimizer manage distribution.

### Use Case 2: Machine-learning model scoring

- **Industry**: Ad technology
- **Scenario**: A scoring job loads a compact model and applies it to billions of impressions.
- **Problem**: Loading the model inside each task repeats I/O and deserialization.
- **Why this concept is relevant**: Broadcasting the model once per executor amortizes the load cost.
- **Implementation idea**: Broadcast the model object and score partitions in a map operation.
- **Expected result**: Faster scoring and less repeated I/O.
- **Alternative**: Load the model from a distributed cache or a sidecar service.
- **Trade-offs**: A sidecar centralizes updates but adds a network dependency and latency.

---

## Business Value

- Lower network and serialization overhead.
- Faster task dispatch for lookup-heavy jobs.
- Simpler sharing of read-only configuration.
- Better use of executor memory for repeated data.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Broadcast only read-only objects that fit comfortably in executor memory.
- Destroy broadcasts when they are no longer needed in long-lived applications.
- Watch driver memory when broadcasting large objects.
- Prefer a broadcast join when the use case is a join.
- Do not mutate the broadcast value inside tasks.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Compare closure capture with broadcast

**Steps**

1. Create a moderately large lookup dictionary and use it inside a map without broadcasting.
2. Broadcast the same dictionary and repeat the job.
3. Compare task serialized size, network traffic, and runtime.

**What to observe**: The closure version sends the dictionary with every task; the broadcast version sends it once per executor.

**Healthy result**: Broadcast reduces task size and runtime.

**Problematic result**: The object is too large and executor memory pressure increases.

**Engineering takeaway**: Broadcast trades driver/executor memory for lower per-task overhead.

---

## Trade-offs and Alternatives

### Advantages

- One transfer per executor
- Peer-to-peer distribution
- Simple API

### Limitations

- Immutable
- Must fit in memory
- Driver holds the source object

### When to use it

- Lookup tables
- Models
- Configuration maps

### When not to use it

- Large mutable state
- Per-task updates
- Broadcasting huge objects

### Alternatives inside Spark

- TorrentBroadcast
- Broadcast joins
- Accumulators

### Alternatives outside Spark

- Distributed caches
- Feature stores
- Model serving

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Broadcast Lookup Set
**Problem Statement**: Broadcast a set of VIP customer IDs.  
**Input / Context**: Set

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Access in DataFrame with UDF
**Problem Statement**: Use broadcast variable inside a Python UDF.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Broadcast Size Limit
**Problem Statement**: Explain driver memory limits on broadcast.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Stale broadcast data

**Scenario**: A lookup table is updated, but a long-running application still uses the old values.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Broadcast variables are immutable and live until destroyed.
2. Check whether the application re-broadcasts after the update.
3. Restart or explicitly destroy and recreate the broadcast.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Broadcast Lookup Set):
```python
b_vips = spark.sparkContext.broadcast({101, 102, 103})
print(101 in b_vips.value)
```

#### Solution 2 (Access in DataFrame with UDF):
```python
@udf(StringType())
def lookup_rate(c):
    return str(b_rates.value.get(c, 1.0))
spark.createDataFrame([('USD',)], ['curr']).select(lookup_rate('curr')).show()
```

#### Solution 3 (Broadcast Size Limit):
```python
print('Broadcast data must fit comfortably in Driver memory and cannot exceed 8GB limit.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Efficiency**: 1 copy per executor JVM, shared by all task threads.
- **Protocol**: BitTorrent-like peer-to-peer distribution.

### Essential Technical Terms
- `Broadcast Variable`: Shared read-only cached object.
- `TorrentBroadcast`: P2P transfer implementation.

### Common Engineering Mistakes
- **Modifying broadcast variable on worker**: Workers cannot mutate broadcast variables; they are strictly read-only.

### Interview Questions on This Topic
- **Q: How does a broadcast variable save memory?**
  - *A*: It copies data once per executor rather than once per task, sharing the memory across all cores on that executor.


### More Interview Questions

- **Q: How does a broadcast variable reduce overhead?**
  - *A*: It sends the object to each executor once and caches it locally, instead of serializing it into every task closure.
- **Q: When should you avoid a broadcast variable?**
  - *A*: When the object is large, mutable, or updated frequently; it must fit in executor and driver memory and cannot be modified in place.

### What to Study Next
Open 07_broadcast_join.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')